# Hotel Booking Demand — Dataset Exploration & Problem Framing

**Dataset:** [Hotel Booking Demand](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand) (Kaggle)

**Goal:** Explore the dataset, frame a business problem, and identify the appropriate ML approach.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('hotel_bookings.csv')
df.shape

(119390, 32)

## 1. Dataset Overview

The dataset contains booking records for a City Hotel and a Resort Hotel in Portugal, covering arrivals between 2015 and 2017. Each row is one booking, with fields describing the guest, the stay, and the booking channel.

In [2]:
df.head(3)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02


## 2. Dataset Shape

In [3]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 119,390
Columns: 32


## 3. Data Types

In [4]:
df.dtypes

hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                     int64
deposit_type                          str
agent                             

## 4. Missing Values

In [5]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})

,missing_count,missing_pct
company,112593,94.31
agent,16340,13.69
country,488,0.41
children,4,0.00


**Notes on missing data:**
- `company` is missing for ~94% of rows — most bookings simply aren't made through a company, so this is expected, not a data quality issue.
- `agent` is missing for ~14% of rows — bookings made without a travel agent (e.g. direct bookings).
- `country` and `children` have negligible missing values (<0.5%) and can be dropped or imputed safely.

## 5. Summary Statistics

In [6]:
df[['lead_time', 'adr', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'children', 'babies', 'total_of_special_requests']].describe().round(2)

,lead_time,adr,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,total_of_special_requests
count,119390.00,119390.00,119390.00,119390.00,119390.00,119386.0,119390.00,119390.00
mean,104.01,101.83,0.93,2.50,1.86,0.1,0.01,0.57
std,106.86,50.54,1.00,1.91,0.58,0.4,0.10,0.79
min,0.00,-6.38,0.00,0.00,0.00,0.0,0.00,0.00
25%,18.00,69.29,0.00,1.00,2.00,0.0,0.00,0.00
50%,69.00,94.58,1.00,2.00,2.00,0.0,0.00,0.00
75%,160.00,126.00,2.00,3.00,2.00,0.0,0.00,1.00
max,737.00,5400.00,19.00,50.00,55.00,10.0,10.00,5.00


**A few things stand out in the summary stats:**
- `adr` (Average Daily Rate) has a minimum of **-6.38** and a maximum of **5400** — both are almost certainly data errors or extreme outliers that should be cleaned before modeling.
- `lead_time` ranges from 0 to 737 days, with a right-skewed distribution (mean ~104 days, median ~69 days).


In [7]:
print("Negative ADR rows:", (df['adr'] < 0).sum())
print("Zero ADR rows:", (df['adr'] == 0).sum())
print("ADR > 1000:", (df['adr'] > 1000).sum())
print("Exact duplicate rows:", df.duplicated().sum())

Negative ADR rows: 1
Zero ADR rows: 1959
ADR > 1000: 1


Exact duplicate rows: 31994


There are also **31,994 exact duplicate rows** in the dataset (about 27% of all rows). This is a known quirk of this dataset — some duplication is plausible (e.g. two identical bookings by coincidence), but this volume suggests real duplication that should be handled (at minimum documented, likely dropped) before any modeling.

## 6. Business Problem

**Problem:** Hotels lose significant revenue when a booking is cancelled close to the arrival date — rooms can go unsold, and cancellations disrupt staffing and inventory planning. If a hotel can predict *at the time of booking* which reservations are likely to be cancelled, it can act early: apply more flexible overbooking strategies, target at-risk bookings with confirmation calls or incentives, or adjust deposit policies.

This is a well-known, high-value problem in the hospitality industry, and this dataset is well suited to it because every row already includes `is_canceled` as a ground-truth label, along with booking-time features (lead time, deposit type, market segment, etc.) that would realistically be known before arrival.

## 7. ML Problem Framing

**Primary task — Classification:** Predict `is_canceled` (binary: 1 = cancelled, 0 = not cancelled) using features available at booking time.

**Justification:** The target variable is categorical (cancelled / not cancelled), not continuous, which rules out regression. There's no natural grouping task here without a label — the business need is a concrete yes/no prediction per booking, not an exploratory grouping — so this is a supervised classification problem rather than clustering.

## 8. Target Variable & Key Features

In [8]:
print("Target variable: is_canceled")
print(df['is_canceled'].value_counts(normalize=True).round(3))

Target variable: is_canceled
is_canceled
0    0.63
1    0.37
Name: proportion, dtype: float64


**Target:** `is_canceled` (0 = not cancelled, 1 = cancelled). The classes are moderately imbalanced (63% / 37%), which should be accounted for in modeling (e.g. class weighting, stratified sampling) but isn't severe enough to require synthetic resampling.

**Key features (all known at booking time, so safe to use for prediction):**
- `lead_time` — days between booking and arrival
- `deposit_type` — No Deposit / Non Refund / Refundable
- `market_segment` — Online TA, Groups, Direct, Corporate, etc.
- `customer_type` — Transient, Group, Contract, Transient-Party
- `previous_cancellations` — guest's cancellation history
- `total_of_special_requests` — proxy for guest engagement/commitment
- `hotel` — City Hotel vs Resort Hotel

Fields like `reservation_status` and `reservation_status_date` are **excluded** from features — they are recorded *after* the outcome is known and would leak the target.

## 9. Key Observations

### Observation 1 — Non-refundable deposits cancel far more often than any other type

In [9]:
df.groupby('deposit_type')['is_canceled'].agg(['mean', 'count']).round(3)

,mean,count
deposit_type,,
No Deposit,0.284,104641
Non Refund,0.994,14587
Refundable,0.222,162


This is the most counterintuitive finding in the dataset. Common sense suggests a non-refundable deposit should *discourage* cancellation (the guest has already paid and stands to lose money). Instead, **Non Refund bookings cancel 99.4% of the time**, compared to just 28.4% for No Deposit bookings. This pattern is well documented for this dataset and is generally attributed to a subset of bookings — likely from certain agents or channels — that are provisional or speculative by nature rather than genuine non-refundable commitments. It's a strong, cheap signal for a cancellation model, but worth flagging as needing investigation rather than taking at face value.

### Observation 2 — Cancellation rate rises steadily and steeply with lead time

In [10]:
df['lead_bucket'] = pd.cut(
    df['lead_time'],
    bins=[-1, 7, 30, 90, 180, 365, 1000],
    labels=['0-7d', '8-30d', '31-90d', '91-180d', '181-365d', '365+d']
)
df.groupby('lead_bucket', observed=True)['is_canceled'].agg(['mean', 'count']).round(3)

,mean,count
lead_bucket,,
0-7d,0.096,19746
8-30d,0.279,18960
31-90d,0.377,29553
91-180d,0.447,26439
181-365d,0.555,21544
365+d,0.677,3148


Cancellation rate climbs almost monotonically with how far in advance a booking is made — from **9.6%** for bookings made within a week of arrival to **67.7%** for bookings made a year or more out. This makes intuitive sense (more time = more opportunity for plans to change) and, unlike deposit type, is a genuinely trustworthy, explainable predictor.

### Observation 3 — Group bookings cancel more than any other market segment, despite seeming like the most committed

In [11]:
df.groupby('market_segment')['is_canceled'].agg(['mean', 'count']).sort_values('count', ascending=False).round(3)

,mean,count
market_segment,,
Online TA,0.367,56477
Offline TA/TO,0.343,24219
Groups,0.611,19811
Direct,0.153,12606
Corporate,0.187,5295
Complementary,0.131,743
Aviation,0.219,237
Undefined,1.000,2


Group bookings cancel at **61.1%** — the highest rate of any market segment with meaningful volume, well above Direct (15.3%) and Corporate (18.7%). This runs against the intuition that group bookings (weddings, tour groups, conferences) represent firmer commitments. In practice, this likely reflects that group bookings are made far in advance and are more exposed to the lead-time effect above, or represent block reservations that get partially or fully released as plans firm up.

## 10. Summary

The Hotel Booking Demand dataset supports a clear, high-value classification problem: predicting booking cancellations using information available at the time of booking. Deposit type, lead time, and market segment all show strong, non-obvious relationships with cancellation, making them promising features for a predictive model — though the deposit type signal in particular needs further investigation rather than being taken at face value.